In [1]:
from moviepy.editor import AudioFileClip, VideoClip, ImageClip
from sqlalchemy import create_engine
from datetime import datetime
import pandas as pd
import os
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# Your database connection
engine = create_engine('mysql+pymysql://root@localhost:3306/music_development')
data_path = '../data/'

# Get song data
sql = '''
SELECT S.id, rank, name, artist_id, A.first_name, A.last_name , youtube_code 
FROM songs S
JOIN artists A
ON S.artist_id = A.id
WHERE LOWER(TRIM(name)) LIKE LOWER(TRIM('Lose Yourself'))
'''
songs = pd.read_sql(sql, engine)
songs

,id,rank,name,artist_id,first_name,last_name,youtube_code
0,667,1173,Lose Yourself,138,Eminem,,7YuAzR2XVAM


In [2]:
songId = songs['id'].iloc[0]
songId

667

In [3]:
sql = f'''
SELECT S.id, rank, name, artist_id, A.first_name, A.last_name , youtube_code 
FROM songs S
JOIN artists A
ON S.artist_id = A.id
WHERE S.id = {songId}
'''
song = pd.read_sql(sql, engine)
song

,id,rank,name,artist_id,first_name,last_name,youtube_code
0,667,1173,Lose Yourself,138,Eminem,,7YuAzR2XVAM


In [4]:
# Get artist name as a string (not pandas Series)
artist_name = f"{song['first_name'].iloc[0]} {song['last_name'].iloc[0]}"
print(f"🎤 Artist: {artist_name}")

🎤 Artist: Eminem 


In [5]:
# Get lyrics
sql = f"""
SELECT id, song_id, content 
FROM lyrics 
WHERE song_id = {songId}
"""
lyrics = pd.read_sql(sql, engine)
lyrics

,id,song_id,content
0,598,667,34|♪ Look ♪\r\n36|♪ If you had ♪\r\n39|♪ One s...


In [6]:
# Get the lyrics content
lyrics_content = lyrics.content.iloc[0]

# Print with proper line breaks (replace \r\n with actual newlines)
print("Full lyrics content:")
print("-" * 50)
print(lyrics_content.replace('\r\n', '\n'))
print("=" * 50)

Full lyrics content:
--------------------------------------------------
34|♪ Look ♪
36|♪ If you had ♪
39|♪ One shot ♪
41|♪ Or one opportunity ♪
43|♪ To seize everything you ever wanted ♪
46|♪ In one moment ♪
48|♪ Would you capture it, or just let it slip? ♪
53|♪ His palms are sweaty ♪
54|♪ Knees weak, arms are heavy ♪
56|♪ There's vomit on his sweater already ♪
58|♪ Mom's spaghetti ♪
59|♪ He's nervous ♪
60|♪ But on the surface he looks calm and ready ♪
62|♪ To drop bombs ♪
63|♪ But he keeps on forgetting ♪
64|♪ What he wrote down ♪
65|♪ The whole crowd goes so loud ♪
67|♪ He opens his mouth, but the words won't come out ♪
70|♪ He's choking, how? Everybody's joking now ♪
73|♪ The clock's run out, time's up, over—blaow! ♪
76|♪ Snap back to reality, ope there goes gravity, ope ♪
79|♪ There goes Rabbit ♪
80|♪ He choked ♪
81|♪ He's so mad but he won't ♪
82|♪ Give up that easy ♪
83|♪ No, he won't have it, he knows ♪
85|♪ His whole back's to these ropes ♪
86|♪ It don't matter, he's dope ♪
87|

In [7]:
def parse_timestamp_lyrics(lyrics_text, font=None, max_width=None):
    """Parse lyrics with timestamp format from YouTube transcripts using pipe separator"""
    lines = [line.strip() for line in lyrics_text.split('\n') if line.strip()]
    
    lyrics_with_timing = []
    
    for line in lines:
        # Split by the first pipe to separate timestamp from text
        if '|' in line:
            timestamp_str, text = line.split('|', 1)
            try:
                # Convert timestamp to float (seconds)
                timestamp = float(timestamp_str.strip())
                lyrics_with_timing.append({
                    'timestamp': timestamp,
                    'text': text.strip(),
                    'start_time': timestamp,
                    'end_time': None,
                    'duration_seconds': None,
                    'word_count': None,
                    'spaces_count': None,
                    'char_count': None
                })
            except ValueError:
                print(f"⚠️ Skipping line with invalid timestamp: {line}")
                continue
        else:
            print(f"⚠️ Skipping line without Pipe separator: {line}")
    
    # Sort by timestamp
    lyrics_with_timing.sort(key=lambda x: x['timestamp'])
    
    # Calculate end times and additional metrics
    for i in range(len(lyrics_with_timing)):
        if i < len(lyrics_with_timing) - 1:
            lyrics_with_timing[i]['end_time'] = lyrics_with_timing[i + 1]['timestamp']
        else:
            lyrics_with_timing[i]['end_time'] = lyrics_with_timing[i]['timestamp'] + 5
        
        lyrics_with_timing[i]['duration_seconds'] = lyrics_with_timing[i]['end_time'] - lyrics_with_timing[i]['start_time']
        
        words = [word for word in lyrics_with_timing[i]['text'].split() if word]
        lyrics_with_timing[i]['word_count'] = len(words)
        lyrics_with_timing[i]['spaces_count'] = max(0, lyrics_with_timing[i]['word_count'] - 1)
        lyrics_with_timing[i]['char_count'] = len(lyrics_with_timing[i]['text'])
        
        # We'll split later after font is loaded in the main function
        lyrics_with_timing[i]['line1'] = lyrics_with_timing[i]['text']
        lyrics_with_timing[i]['line2'] = ''
        lyrics_with_timing[i]['is_split'] = False
    
    return lyrics_with_timing

def split_text_to_fit_width(text, font, max_width, draw):
    """Split text into multiple lines to fit within max_width"""
    if not text:
        return ['']
    
    # Check if text fits in one line
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
    except AttributeError:
        # For older PIL versions
        text_width, text_height = draw.textsize(text, font=font)
    else:
        text_width = bbox[2] - bbox[0]
    
    if text_width <= max_width:
        return [text]
    
    # Text doesn't fit - need to split
    words = text.split()
    lines = []
    current_line = []
    
    for word in words:
        # Try adding the word to current line
        test_line = ' '.join(current_line + [word])
        try:
            bbox = draw.textbbox((0, 0), test_line, font=font)
        except AttributeError:
            test_width, _ = draw.textsize(test_line, font=font)
        else:
            test_width = bbox[2] - bbox[0]
        
        if test_width <= max_width:
            # Word fits, add it
            current_line.append(word)
        else:
            # Word doesn't fit
            if current_line:
                # Save current line and start new one with this word
                lines.append(' '.join(current_line))
                current_line = [word]
            else:
                # Single word is too long - need character split
                # Find where to split the word
                for i in range(len(word), 0, -1):
                    test_part = word[:i]
                    try:
                        bbox = draw.textbbox((0, 0), test_part, font=font)
                    except AttributeError:
                        test_width, _ = draw.textsize(test_part, font=font)
                    else:
                        test_width = bbox[2] - bbox[0]
                    
                    if test_width <= max_width:
                        lines.append(word[:i])
                        current_line = [word[i:]]
                        break
                else:
                    # If even single character doesn't fit, just add the word
                    lines.append(word)
                    current_line = []
    
    # Add remaining words
    if current_line:
        lines.append(' '.join(current_line))
    
    # Limit to 2 lines for this application
    if len(lines) > 2:
        # If more than 2 lines, combine the rest with ellipsis
        lines = lines[:2]
        if len(lines[1]) > max_width * 0.8:  # If second line is long, add ellipsis
            lines[1] = lines[1][:int(len(lines[1]) * 0.8)] + "..."
    
    return lines

def get_current_lyric(current_time, lyrics_with_timing):
    """Find which lyric should be displayed at current time"""
    for lyric in lyrics_with_timing:
        if lyric['start_time'] <= current_time < lyric['end_time']:
            return lyric
    return None

def create_timestamp_synced_video(song_id=songId, max_duration=None):
    """Create video using timestamp-based lyrics from YouTube transcripts"""
    
    try:
        # Get song data - use %s parameter syntax for PyMySQL
        query = """
        SELECT s.name as song_name, s.location as audio_file,
               l.content as lyrics, a.first_name, a.last_name
        FROM songs s 
        JOIN lyrics l ON s.id = l.song_id 
        JOIN artists a ON s.artist_id = a.id 
        WHERE s.id = %s
        """
        
        df = pd.read_sql(query, engine, params=(song_id,))
        
        if df.empty:
            print(f"❌ No song found with ID: {song_id}")
            return None
            
        song_data = df.iloc[0]
        
        print(f"🎵 Creating TIMESTAMP-SYNCED video for: {song_data['song_name']}")
        
        # Construct file paths
        audio_dir = os.path.join(r"C:\ruby\music\public\uploads\song\location", str(song_id))
        
        if os.path.exists(audio_dir):
            print(f"📁 Directory contents: {os.listdir(audio_dir)}")
        else:
            print(f"❌ Directory doesn't exist: {audio_dir}")
            return None
            
        audio_path = os.path.join(audio_dir, song_data['audio_file'])
        
        print(f"🔊 Audio path: {audio_path}")
        print(f"🔊 Audio file from DB: {song_data['audio_file']}")

        background_image_path = os.path.join(audio_dir, "Folder.jpg")
        print(f"🖼️ Background image: {background_image_path}")        
        
        if not os.path.exists(audio_path):
            print(f"❌ Audio file not found at: {audio_path}")
            all_files = os.listdir(audio_dir)
            print(f"📁 Available files: {all_files}")
            for file in all_files:
                if "I_Don" in file and "Want_to_Talk" in file:
                    print(f"🔍 Found alternative file: {file}")
                    audio_path = os.path.join(audio_dir, file)
                    break
            else:
                return None
        
        # Video settings (defined early for text width calculation)
        fps = 24
        width, height = 640, 480
        
        # Load fonts early for text width calculation
        try:
            font_main = ImageFont.truetype("arial.ttf", 32)
        except:
            try:
                font_main = ImageFont.truetype("C:/Windows/Fonts/arial.ttf", 32)
            except:
                font_main = ImageFont.load_default()
        
        # Create a temporary draw object for text measurement
        temp_img = Image.new('RGB', (width, height), color='black')
        temp_draw = ImageDraw.Draw(temp_img)
        
        # Define margins (text won't go edge to edge)
        left_margin = 40
        right_margin = 40
        max_text_width = width - left_margin - right_margin
        
        print(f"📏 Max text width: {max_text_width}px (with {left_margin}px margins)")
        
        # Parse timestamp-based lyrics
        lyrics_with_timing = parse_timestamp_lyrics(song_data['lyrics'])
        
        if not lyrics_with_timing:
            print("❌ Could not parse timestamp lyrics")
            return None
        
        # Now split long lines based on actual font metrics
        split_count = 0
        for lyric in lyrics_with_timing:
            text = lyric['text']
            lines = split_text_to_fit_width(text, font_main, max_text_width, temp_draw)
            
            if len(lines) > 1:
                lyric['line1'] = lines[0]
                lyric['line2'] = lines[1] if len(lines) > 1 else ''
                lyric['is_split'] = True
                split_count += 1
            else:
                lyric['line1'] = lines[0]
                lyric['line2'] = ''
                lyric['is_split'] = False
        
        print(f"📝 Parsed {len(lyrics_with_timing)} timestamped lyrics lines")
        print(f"✂️ Split {split_count} lines that would overflow")
        
        # Load audio clip
        audio_clip = AudioFileClip(audio_path)
        full_duration = audio_clip.duration
        
        if max_duration:
            duration = min(max_duration, full_duration)
            print(f"⏱️ Using LIMITED duration: {duration:.1f}s (max_duration={max_duration}s)")
        else:
            duration = full_duration
            print(f"⏱️ Using FULL duration: {duration:.1f}s")
        
        if max_duration and full_duration > max_duration:
            audio_clip = audio_clip.subclip(0, duration)
        
        if max_duration:
            lyrics_with_timing = [lyric for lyric in lyrics_with_timing if lyric['start_time'] < duration]
            if lyrics_with_timing and lyrics_with_timing[-1]['end_time'] > duration:
                lyrics_with_timing[-1]['end_time'] = duration
        
        print(f"📝 Final timing: {len(lyrics_with_timing)} lyrics lines")
        print(f"⏱️ Video duration: {duration:.1f}s ({duration/60:.1f} minutes)")

        print("\n📋 Parsed Lyrics Timing:")
        print("-" * 90)
        for i, lyric in enumerate(lyrics_with_timing):
            display_text = lyric['text'][:45] + ('...' if len(lyric['text']) > 45 else '')
            split_indicator = "✂️" if lyric['is_split'] else "  "
            
            # Calculate approximate width percentage for info
            try:
                bbox = temp_draw.textbbox((0, 0), lyric['text'], font=font_main)
                text_width = bbox[2] - bbox[0]
                width_percent = (text_width / max_text_width) * 100
                width_info = f"{width_percent:3.0f}%"
            except:
                width_info = "N/A"
            
            print(f"{i+1:2d}. {split_indicator} {lyric['start_time']:5.1f}s - {lyric['end_time']:5.1f}s: "
                  f"{display_text:<47} | {lyric['duration_seconds']:4.1f}s | {width_info}")
        
        def make_synced_frame(t):
            try:
                # Load background
                if os.path.exists(background_image_path):
                    bg_image = Image.open(background_image_path)
                    bg_image = bg_image.resize((width, height), Image.Resampling.LANCZOS)
                    frame = np.array(bg_image)
                else:
                    frame = np.full((height, width, 3), [40, 40, 80], dtype=np.uint8)
                
                pil_img = Image.fromarray(frame)
                draw = ImageDraw.Draw(pil_img)
                
                # Get current lyric
                current_lyric = get_current_lyric(t, lyrics_with_timing)
                
                if current_lyric:
                    text_color = (255, 255, 255)  # White
                    
                    if current_lyric['is_split']:
                        # Draw two lines
                        line1 = current_lyric['line1']
                        line2 = current_lyric['line2']
                        
                        # Calculate positions for both lines
                        try:
                            bbox1 = draw.textbbox((0, 0), line1, font=font_main)
                            bbox2 = draw.textbbox((0, 0), line2, font=font_main)
                        except AttributeError:
                            text_width1, text_height1 = draw.textsize(line1, font=font_main)
                            text_width2, text_height2 = draw.textsize(line2, font=font_main)
                            bbox1 = (0, 0, text_width1, text_height1)
                            bbox2 = (0, 0, text_width2, text_height2)
                        
                        text_width1 = bbox1[2] - bbox1[0]
                        text_height1 = bbox1[3] - bbox1[1]
                        text_width2 = bbox2[2] - bbox2[0]
                        text_height2 = bbox2[3] - bbox2[1]
                        
                        # Center both lines horizontally
                        x1 = (width - text_width1) // 2
                        x2 = (width - text_width2) // 2
                        
                        # Position lines vertically
                        line_spacing = 8  # Space between lines
                        y_start = height - 60  # Base position from bottom for the SECOND line
                        
                        # Position the second line at y_start
                        y2 = y_start
                        # Position the first line above it
                        y1 = y2 - text_height2 - line_spacing
                        
                        # Add semi-transparent background for better readability (optional)
                        if False:  # Set to True to enable background box
                            padding = 10
                            box_coords = [
                                min(x1, x2) - padding,
                                y1 - padding,
                                max(x1 + text_width1, x2 + text_width2) + padding,
                                y2 + text_height2 + padding
                            ]
                            overlay = Image.new('RGBA', pil_img.size, (0, 0, 0, 0))
                            overlay_draw = ImageDraw.Draw(overlay)
                            overlay_draw.rectangle(box_coords, fill=(0, 0, 0, 128))
                            pil_img = Image.alpha_composite(pil_img.convert('RGBA'), overlay)
                            draw = ImageDraw.Draw(pil_img)
                        
                        # Draw both lines
                        draw.text((x1, y1), line1, font=font_main, fill=text_color)
                        draw.text((x2, y2), line2, font=font_main, fill=text_color)
                    else:
                        # Draw single line - POSITIONED AT LOWER LINE POSITION
                        current_line = current_lyric['text']
                        
                        try:
                            bbox = draw.textbbox((0, 0), current_line, font=font_main)
                        except AttributeError:
                            text_width, text_height = draw.textsize(current_line, font=font_main)
                            bbox = (0, 0, text_width, text_height)
                        
                        text_width = bbox[2] - bbox[0]
                        text_height = bbox[3] - bbox[1]
                        
                        # Center horizontally
                        x = (width - text_width) // 2
                        
                        # Position at the same height as the second line of split lyrics
                        y_start = height - 60  # Base position from bottom
                        y = y_start  # Position single line exactly where the second line would be
                        
                        draw.text((x, y), current_line, font=font_main, fill=text_color)
                
                return np.array(pil_img)
                
            except Exception as e:
                print(f"❌ Frame error at {t:.1f}s: {e}")
                return np.zeros((height, width, 3), dtype=np.uint8)
        
        # Create video
        print("🎬 Creating timestamp-synced video frames...")
        video = VideoClip(make_synced_frame, duration=duration)
        video = video.set_audio(audio_clip)
        
        # Export
        output_dir = '../data/videos'
        os.makedirs(output_dir, exist_ok=True)
        
        clean_artist_name = f"{song_data['first_name']} {song_data['last_name']}"
        output_file = os.path.join(output_dir, f"{song_data['song_name']}_{clean_artist_name}.mp4")
        
        print("📹 Exporting video...")
        video.write_videofile(
            output_file, 
            fps=fps, 
            codec='libx264',
            audio_codec='aac',
            verbose=False,
            logger=None
        )
        
        print(f"✅ TIMESTAMP-SYNCED VIDEO CREATED: {output_file}")
        print(f"📊 File size: {os.path.getsize(output_file) / (1024*1024):.1f} MB")
        
        # Clean up
        video.close()
        audio_clip.close()
        
        return output_file
        
    except Exception as e:
        print(f"❌ Video creation error: {e}")
        import traceback
        traceback.print_exc()
        return None

# Run the timestamp-synced version
if __name__ == "__main__":
    print("=" * 70)
    print("🎬 CREATING TIMESTAMP-SYNCED VIDEO")
    print("=" * 70)
    
    result = create_timestamp_synced_video(song_id=songId, max_duration=None)
    
    if result:
        print(f"\n🎉 SUCCESS! Timestamp-synced video created: {result}")
        print("\n✨ Features:")
        print("   ✅ YouTube transcript timestamp parsing")
        print("   ✅ Intelligent text overflow detection")
        print("   ✅ Dynamic line splitting based on font metrics")
        print("   ✅ Single lines positioned at lower line height for consistency")
        print("   ✅ Perfect synchronization with original timestamps")
        
    else:
        print("\n❌ Timestamp-synced video creation failed")

🎬 CREATING TIMESTAMP-SYNCED VIDEO
🎵 Creating TIMESTAMP-SYNCED video for: Lose Yourself
📁 Directory contents: ['Eminem_-_Lose_Yourself.mp3', 'Folder.jpg']
🔊 Audio path: C:\ruby\music\public\uploads\song\location\667\Eminem_-_Lose_Yourself.mp3
🔊 Audio file from DB: Eminem_-_Lose_Yourself.mp3
🖼️ Background image: C:\ruby\music\public\uploads\song\location\667\Folder.jpg
📏 Max text width: 560px (with 40px margins)
📝 Parsed 129 timestamped lyrics lines
✂️ Split 52 lines that would overflow
⏱️ Using FULL duration: 328.3s
📝 Final timing: 129 lyrics lines
⏱️ Video duration: 328.3s (5.5 minutes)

📋 Parsed Lyrics Timing:
------------------------------------------------------------------------------------------
 1.     34.0s -  36.0s: ♪ Look ♪                                        |  2.0s |  21%
 2.     36.0s -  39.0s: ♪ If you had ♪                                  |  3.0s |  34%
 3.     39.0s -  41.0s: ♪ One shot ♪                                    |  2.0s |  32%
 4.     41.0s -  43.0s: ♪ Or 